# mbo viewer window config

Config is read when the figure is built, so set it before `run_gui`. Windows already on screen are changed through the live objects. `vis.close()` before opening another.

In [38]:
import fastplotlib as fpl
from fastplotlib import global_config
from fastplotlib.layouts import Subplot
from mbo_utilities import DataVis
from mbo_utilities.gui.run_gui import run_gui

DATA = r"X:\data\eunji\raw"

print(Subplot.config.init)
print(Subplot.config.auto_scale)

init(toolbar=False, background_color='k', frame_kwargs={'spacing': {'x0': 4, 'sides': 6, 'title_flanks': 10, 'resize_handle_space': 10, 'bottom': 10}, 'title_kwargs': {'font_size': 8, 'face_color': 'w'}, 'plane_color': None})
auto_scale(maintain_aspect=None, zoom=1.99)


## before opening: padding, title, zoom

Padding is the subplot frame spacing. `update` merges dicts key by key, so only the keys named here change. `Subplot.config.init.toolbar` does not reach this viewer: PreviewDataWidget switches every toolbar off when it starts, so toolbars are set on the live figure below.

In [60]:
global_config.update(
    Subplot.config.init,
    frame_kwargs={
        "spacing": {"x0": 4, "sides": 6, "title_flanks": 10, "resize_handle_space": 10, "bottom": 10},
        "title_kwargs": {"font_size": 12},
    },
)
Subplot.config.auto_scale.zoom = 0.99
fig.canvas.set_logical_size(2000, 800)

vis = run_gui(DATA)

INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.
INFO - mbo.reader - Detected zarr.json, loading as ZarrArray.


RFBOutputContext()

In [ ]:
vis.iw.figure.canvas.set_logical_size(2000, 800)

## the windows

The figure keeps its edge windows by location: `right` is PreviewDataWidget, `bottom` the slider bar, `top` the menu strip. `size` is the thickness the window reserves, so shutting a window means size and collapsed together.

In [40]:
fig = vis.iw.figure
side, bar, strip = fig.imgui_windows["right"], fig.imgui_windows["bottom"], fig.imgui_windows["top"]

{name: (w.size, w.collapsed) for name, w in fig.imgui_windows.items() if w is not None}

{'right': (300, False), 'top': (478, False), 'bottom': (0, True)}

## side panel width

In [41]:
side.size = 220

## shut everything

In [42]:
side.size = 0
side.collapsed = True
bar.size = 0
bar.collapsed = True
strip.toggle_collapsed()  # only the menu row stays
for subplot in fig:
    subplot.toolbar = False

## open everything, toolbars on

The slider bar sizes itself to its slider count. The strip goes back to sizing itself to its panel, or `strip.resize_to(200)` pins its height.

In [43]:
side.size = 300
side.collapsed = False
bar.size = 57 + 50 * vis.iw.n_sliders
bar.collapsed = False
strip.reset_size()
for subplot in fig:
    subplot.toolbar = True

## wider

The canvas resizes live, in a notebook cell and in a desktop window alike. The edge windows keep their thickness, so all the extra room goes to the image. The top strip is the exception: it grows with canvas height unless it is shut or pinned.

In [48]:
strip.toggle_collapsed()  # menu row only
fig.canvas.set_logical_size(2000, 800)

## fit the canvas to the image

Sizes the canvas so the image fills its area edge to edge at `image_width` pixels: the edge windows, the histogram beside the image, and the frame padding are added around it. Shut the strip first, or it is pinned at its current height. Written for the single subplot viewer.

In [ ]:
def fit_canvas(vis, image_width=800):
    fig = vis.iw.figure
    subplot = fig[0, 0]
    rows, cols = vis.iw.data[0].shape[-2:]
    strip = fig.imgui_windows["top"]
    if not strip.collapsed:
        strip.resize_to(strip.size)  # otherwise it grows with the canvas
    edges = {k: w.size for k, w in fig.imgui_windows.items() if w is not None}
    inner = {k: w.size for k, w in subplot.imgui_windows.items() if w is not None and k != "toolbar"}
    fx, fy, fw, fh = subplot.frame.rect
    rx, ry, rw, rh = subplot.frame.get_render_rect()
    chrome_w = (fw - rw) + inner.get("left", 0) + inner.get("right", 0)
    chrome_h = (fh - rh) + inner.get("top", 0) + inner.get("bottom", 0)
    width = image_width + chrome_w + edges.get("left", 0) + edges.get("right", 0)
    height = image_width * rows / cols + chrome_h + edges.get("top", 0) + edges.get("bottom", 0)
    fig.canvas.set_logical_size(round(width), round(height))


fit_canvas(vis, image_width=800)

## compact preset

`flynn` is `very_compact` plus auto-scale zoom 0.99: no toolbar, no frame spacing, no title. Presets are additive, so set what they leave alone yourself.

In [ ]:
vis.close()

fpl.presets.flynn()
vis = run_gui(DATA)

## canvas size at open

`run_gui` sizes the canvas from the screen, which bypasses `Figure.config.init.size`. Pass it to `DataVis` instead.

In [ ]:
vis.close()

vis = DataVis(DATA, size=(1200, 800))
vis.show()

## one function for the layout you settle on

Config before opening, live windows after. Fit in the next cell: the slider bar sizes itself on its next frame, so a fit in the same cell would use the old height.

In [ ]:
def my_layout(vis, side_width=280, toolbars=False, sliders=True):
    fig = vis.iw.figure
    panel, bar, strip = fig.imgui_windows["right"], fig.imgui_windows["bottom"], fig.imgui_windows["top"]
    panel.size = side_width
    panel.collapsed = side_width == 0
    bar.size = 57 + 50 * vis.iw.n_sliders if sliders else 0
    bar.collapsed = not sliders
    if not strip.collapsed:
        strip.toggle_collapsed()
    for subplot in fig:
        subplot.toolbar = toolbars


vis.close()

fpl.presets.flynn()
vis = run_gui(DATA)
my_layout(vis, side_width=260, toolbars=False, sliders=True)

In [ ]:
fit_canvas(vis, image_width=800)

## back to defaults

In [ ]:
vis.close()

fpl.presets.default()
print(Subplot.config.init)
print(Subplot.config.auto_scale)